# Class Imbalance Analysis for Grade + Time Model

This notebook analyzes:
1. **Grade distribution** - Class imbalance across grades
2. **Time delta distribution** - Regression target analysis
3. **Dilution-specific balance** - 10% vs 3% dilution patterns
4. **Joint distribution** - Grade × Time Delta interactions
5. **Balancing strategies** - Class weights, sampling, and augmentation

**Dataset:** Optimal time dataset (train_optimal.csv, val_optimal.csv, test_optimal.csv)

**Author:** Sayumi Devasurendra  
**Date:** 2025-12-30  
**Version:** 2.0 (Updated for optimal time approach)

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from collections import Counter
import json

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from utils.class_balance_utils import (
    calculate_class_weights,
    create_weighted_sampler,
    create_joint_weighted_sampler,
    analyze_class_distribution,
    analyze_time_delta_distribution,
    analyze_dilution_distribution,
    print_class_distribution,
    print_time_delta_distribution
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Create output directory
output_dir = Path('../results/results_04/figures/class_imbalance_analysis')
output_dir.mkdir(parents=True, exist_ok=True)

print("✓ Imports successful!")
print(f"✓ Output directory: {output_dir}")

## 2. Load Dataset

In [ ]:
# Load optimal time datasets
splits_dir = Path.cwd().parent / 'data' / 'data_04' / 'splits'

train_df = pd.read_csv(splits_dir / 'train_optimal.csv')
val_df = pd.read_csv(splits_dir / 'val_optimal.csv')
test_df = pd.read_csv(splits_dir / 'test_optimal.csv')

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)
print(f"Train samples:      {len(train_df):4d} ({len(train_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}%)")
print(f"Validation samples: {len(val_df):4d} ({len(val_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}%)")
print(f"Test samples:       {len(test_df):4d} ({len(test_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}%)")
print(f"\nTotal: {len(train_df) + len(val_df) + len(test_df)} samples")

# Show columns
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nFirst few rows:")
display(train_df.head())

---
# Part I: Grade Distribution Analysis

## 3. Grade Class Distribution

In [ ]:
# Analyze grade distribution
train_stats = analyze_class_distribution(train_df)
val_stats = analyze_class_distribution(val_df)
test_stats = analyze_class_distribution(test_df)

print_class_distribution(train_stats, "Training Set Grade Distribution")
print("\n")
print_class_distribution(val_stats, "Validation Set Grade Distribution")
print("\n")
print_class_distribution(test_stats, "Test Set Grade Distribution")

In [ ]:
# Visualize grade distribution across splits
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#e74c3c', '#f39c12', '#2ecc71', '#9b59b6']

for idx, (df, name, stats) in enumerate([
    (train_df, 'Train', train_stats),
    (val_df, 'Validation', val_stats),
    (test_df, 'Test', test_stats)
]):
    grades = sorted(stats['distribution'].keys())
    counts = [stats['distribution'][g] for g in grades]
    
    axes[idx].bar(grades, counts, color=colors)
    axes[idx].set_xlabel('Grade', fontsize=12)
    axes[idx].set_ylabel('Count', fontsize=12)
    axes[idx].set_title(f'{name} Set\n(n={len(df)})', fontsize=14, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Add count labels
    for grade, count in zip(grades, counts):
        axes[idx].text(grade, count + 1, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(output_dir / '01_grade_distribution_by_split.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {output_dir / '01_grade_distribution_by_split.png'}")

## 4. Grade Distribution by Dilution

In [ ]:
# Analyze dilution-specific grade distribution
dilution_stats = analyze_dilution_distribution(train_df)

print("=" * 70)
print("DILUTION DISTRIBUTION")
print("=" * 70)
for dil, count in dilution_stats['distribution'].items():
    pct = dilution_stats['percentages'][dil]
    print(f"{dil}: {count:3d} samples ({pct:.1f}%)")

# Grade distribution per dilution
print("\n" + "=" * 70)
print("GRADE DISTRIBUTION BY DILUTION")
print("=" * 70)

for dilution in sorted(train_df['dilution'].unique()):
    subset = train_df[train_df['dilution'] == dilution]
    stats = analyze_class_distribution(subset)
    print(f"\n{dilution} Dilution (n={len(subset)}):")
    for grade in sorted(stats['distribution'].keys()):
        count = stats['distribution'][grade]
        pct = stats['percentages'][grade]
        print(f"  Grade {grade}: {count:3d} ({pct:5.1f}%)")

In [ ]:
# Visualize grade distribution by dilution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, dilution in enumerate(sorted(train_df['dilution'].unique())):
    subset = train_df[train_df['dilution'] == dilution]
    grade_counts = subset['grade_numeric'].value_counts().sort_index()
    
    axes[idx].bar(grade_counts.index, grade_counts.values, color=colors)
    axes[idx].set_xlabel('Grade', fontsize=12)
    axes[idx].set_ylabel('Count', fontsize=12)
    axes[idx].set_title(f'{dilution} Dilution\n(n={len(subset)})', fontsize=14, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Add count labels
    for grade, count in zip(grade_counts.index, grade_counts.values):
        axes[idx].text(grade, count + 1, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(output_dir / '02_grade_distribution_by_dilution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {output_dir / '02_grade_distribution_by_dilution.png'}")

---
# Part II: Time Delta Distribution Analysis

## 5. Time Delta Statistics

In [ ]:
# Analyze time delta distribution
train_time_stats = analyze_time_delta_distribution(train_df)
val_time_stats = analyze_time_delta_distribution(val_df)
test_time_stats = analyze_time_delta_distribution(test_df)

print_time_delta_distribution(train_time_stats, "Training Set Time Delta Distribution")
print("\n")
print_time_delta_distribution(val_time_stats, "Validation Set Time Delta Distribution")
print("\n")
print_time_delta_distribution(test_time_stats, "Test Set Time Delta Distribution")

In [ ]:
# Visualize time delta distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Histogram
axes[0, 0].hist(train_df['time_delta'], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Optimal (Δ=0)')
axes[0, 0].set_xlabel('Time Delta (minutes)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('Time Delta Distribution (Training Set)', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Box plot by dilution
train_df.boxplot(column='time_delta', by='dilution', ax=axes[0, 1])
axes[0, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Dilution', fontsize=12)
axes[0, 1].set_ylabel('Time Delta (minutes)', fontsize=12)
axes[0, 1].set_title('Time Delta by Dilution', fontsize=14, fontweight='bold')
axes[0, 1].get_figure().suptitle('')  # Remove default title
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Time delta categories
categories = ['At Optimal\n(Δ=0)', 'Before Optimal\n(Δ>0)', 'After Optimal\n(Δ<0)']
counts = [
    train_time_stats['at_optimal_count'],
    train_time_stats['before_optimal_count'],
    train_time_stats['after_optimal_count']
]
cat_colors = ['#2ecc71', '#3498db', '#e74c3c']

axes[1, 0].bar(categories, counts, color=cat_colors)
axes[1, 0].set_ylabel('Count', fontsize=12)
axes[1, 0].set_title('Timing Categories', fontsize=14, fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

for i, (cat, count) in enumerate(zip(categories, counts)):
    axes[1, 0].text(i, count + 5, str(count), ha='center', fontweight='bold')

# Plot 4: Cumulative distribution
sorted_deltas = np.sort(train_df['time_delta'].values)
cumulative = np.arange(1, len(sorted_deltas) + 1) / len(sorted_deltas) * 100

axes[1, 1].plot(sorted_deltas, cumulative, linewidth=2, color='#9b59b6')
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Optimal')
axes[1, 1].set_xlabel('Time Delta (minutes)', fontsize=12)
axes[1, 1].set_ylabel('Cumulative Percentage (%)', fontsize=12)
axes[1, 1].set_title('Cumulative Distribution', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / '03_time_delta_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {output_dir / '03_time_delta_analysis.png'}")

## 6. Joint Distribution: Grade × Time Delta

In [ ]:
# Create heatmap of grade vs time_delta
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap 1: Count-based
pivot_count = train_df.pivot_table(
    index='grade_numeric',
    columns=pd.cut(train_df['time_delta'], bins=[-np.inf, -5, -2, 0, 2, 5, np.inf]),
    values='filename',
    aggfunc='count',
    fill_value=0
)

sns.heatmap(pivot_count, annot=True, fmt='g', cmap='YlOrRd', ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_xlabel('Time Delta Range (minutes)', fontsize=12)
axes[0].set_ylabel('Grade', fontsize=12)
axes[0].set_title('Grade × Time Delta Distribution (Counts)', fontsize=14, fontweight='bold')

# Scatter plot: Grade vs Time Delta
for grade in sorted(train_df['grade_numeric'].unique()):
    subset = train_df[train_df['grade_numeric'] == grade]
    axes[1].scatter(subset['time_delta'], subset['grade_numeric'], 
                   alpha=0.4, s=50, label=f'Grade {grade}')

axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Optimal')
axes[1].set_xlabel('Time Delta (minutes)', fontsize=12)
axes[1].set_ylabel('Grade', fontsize=12)
axes[1].set_title('Grade vs Time Delta Scatter', fontsize=14, fontweight='bold')
axes[1].set_yticks([2, 3, 4, 5])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / '04_grade_time_delta_joint.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {output_dir / '04_grade_time_delta_joint.png'}")

---
# Part III: Class Balancing Strategies

## 7. Calculate Class Weights

In [ ]:
# Convert labels to 0-indexed
train_labels = train_df['grade_numeric'].values - 1

# Calculate weights using different methods
weights_inverse = calculate_class_weights(train_labels, num_classes=5, method='inverse')
weights_effective = calculate_class_weights(train_labels, num_classes=5, method='effective')

print("=" * 70)
print("CLASS WEIGHTS FOR LOSS FUNCTION")
print("=" * 70)
print(f"{'Grade':<10} {'Count':<10} {'Inverse':<15} {'Effective':<15}")
print("-" * 70)

for i in range(5):
    grade_num = i + 1
    count = train_stats['distribution'].get(grade_num, 0)
    print(f"Grade {grade_num:<4} {count:<10} {weights_inverse[i]:<15.4f} {weights_effective[i]:<15.4f}")

print("=" * 70)
print("\nRecommendation: Use 'inverse' weights for CrossEntropyLoss")

In [ ]:
# Visualize class weights
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

grades_plot = [1, 2, 3, 4, 5]
x = np.arange(len(grades_plot))
width = 0.35

# Plot 1: Weight comparison
axes[0].bar(x - width/2, weights_inverse.numpy(), width, label='Inverse Frequency', color='#4ecdc4')
axes[0].bar(x + width/2, weights_effective.numpy(), width, label='Effective Number', color='#45b7d1')
axes[0].set_xlabel('Grade', fontsize=12)
axes[0].set_ylabel('Weight', fontsize=12)
axes[0].set_title('Class Weight Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(grades_plot)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Sample counts vs weights
sample_counts = [train_stats['distribution'].get(g, 0) for g in grades_plot]

ax2 = axes[1]
ax2_twin = ax2.twinx()

bars = ax2.bar(x, sample_counts, color='#e74c3c', alpha=0.6, label='Sample Count')
line = ax2_twin.plot(x, weights_inverse.numpy(), 'o-', color='#2ecc71', linewidth=2, markersize=8, label='Weight')

ax2.set_xlabel('Grade', fontsize=12)
ax2.set_ylabel('Sample Count', fontsize=12, color='#e74c3c')
ax2_twin.set_ylabel('Weight', fontsize=12, color='#2ecc71')
ax2.set_title('Sample Count vs Class Weight', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(grades_plot)
ax2.grid(axis='y', alpha=0.3)

# Add legend
lines = [bars, line[0]]
labels = ['Sample Count', 'Inverse Weight']
ax2.legend(lines, labels, loc='upper left')

plt.tight_layout()
plt.savefig(output_dir / '05_class_weights.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {output_dir / '05_class_weights.png'}")

## 8. Test Weighted Sampling

In [ ]:
# Test simple grade-based sampler
simple_sampler = create_weighted_sampler(train_labels, num_classes=5, method='inverse')

# Simulate sampling
num_samples = 1000
sampled_indices = list(simple_sampler)[:num_samples]
sampled_labels = train_labels[sampled_indices]

# Count distribution
original_dist = Counter(train_labels)
sampled_dist = Counter(sampled_labels)

print("=" * 70)
print("SIMPLE WEIGHTED SAMPLING (Grade Only)")
print("=" * 70)
print(f"{'Grade':<10} {'Original':<15} {'Sampled':<15} {'Change':<15}")
print("-" * 70)

for grade_idx in range(5):
    grade_num = grade_idx + 1
    orig_count = original_dist.get(grade_idx, 0)
    samp_count = sampled_dist.get(grade_idx, 0)
    orig_pct = (orig_count / len(train_labels)) * 100
    samp_pct = (samp_count / num_samples) * 100
    diff = samp_pct - orig_pct
    
    print(f"Grade {grade_num:<4} {orig_pct:<14.2f}% {samp_pct:<14.2f}% {diff:+.2f}%")
    
print("=" * 70)

In [ ]:
# Test joint sampler (grade + dilution + time_delta)
joint_sampler = create_joint_weighted_sampler(
    train_df,
    grade_weight=0.5,
    dilution_weight=0.3,
    time_delta_weight=0.2
)

# Simulate sampling
joint_sampled_indices = list(joint_sampler)[:num_samples]
joint_sampled_df = train_df.iloc[joint_sampled_indices]

print("\n=" * 70)
print("JOINT WEIGHTED SAMPLING (Grade + Dilution + Time Delta)")
print("=" * 70)

# Grade distribution
print("\nGrade Distribution:")
joint_grade_dist = joint_sampled_df['grade_numeric'].value_counts().sort_index()
for grade in sorted(train_df['grade_numeric'].unique()):
    orig_pct = (train_df['grade_numeric'] == grade).sum() / len(train_df) * 100
    joint_pct = (joint_sampled_df['grade_numeric'] == grade).sum() / len(joint_sampled_df) * 100
    print(f"  Grade {grade}: {orig_pct:5.1f}% → {joint_pct:5.1f}% ({joint_pct-orig_pct:+.1f}%)")

# Dilution distribution
print("\nDilution Distribution:")
for dil in sorted(train_df['dilution'].unique()):
    orig_pct = (train_df['dilution'] == dil).sum() / len(train_df) * 100
    joint_pct = (joint_sampled_df['dilution'] == dil).sum() / len(joint_sampled_df) * 100
    print(f"  {dil}: {orig_pct:5.1f}% → {joint_pct:5.1f}% ({joint_pct-orig_pct:+.1f}%)")

# Time delta statistics
print("\nTime Delta Statistics:")
print(f"  Original mean: {train_df['time_delta'].mean():+.2f} min")
print(f"  Sampled mean: {joint_sampled_df['time_delta'].mean():+.2f} min")
print(f"  Original std: {train_df['time_delta'].std():.2f} min")
print(f"  Sampled std: {joint_sampled_df['time_delta'].std():.2f} min")

print("=" * 70)

---
# Part IV: Summary and Recommendations

## 9. Summary Statistics

In [ ]:
print("=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print("\n📊 Dataset Size:")
print(f"  Train: {len(train_df)} samples")
print(f"  Val: {len(val_df)} samples")
print(f"  Test: {len(test_df)} samples")
print(f"  Total: {len(train_df) + len(val_df) + len(test_df)} samples")

print("\n📈 Grade Imbalance (Training):")
print(f"  Most common: Grade {train_stats['max_class']} ({train_stats['max_count']} samples)")
print(f"  Least common: Grade {train_stats['min_class']} ({train_stats['min_count']} samples)")
print(f"  Imbalance ratio: {train_stats['imbalance_ratio']:.2f}x")
print(f"  Balance ratio: {train_stats['balance_ratio']:.2f}")

print("\n⏱️ Time Delta Statistics (Training):")
print(f"  Mean: {train_time_stats['mean']:+.2f} min")
print(f"  Median: {train_time_stats['median']:+.2f} min")
print(f"  Range: [{train_time_stats['min']:+.1f}, {train_time_stats['max']:+.1f}] min")
print(f"  At optimal: {train_time_stats['at_optimal_count']} ({train_time_stats['at_optimal_pct']:.1f}%)")
print(f"  Before optimal: {train_time_stats['before_optimal_count']} ({train_time_stats['before_optimal_pct']:.1f}%)")
print(f"  After optimal: {train_time_stats['after_optimal_count']} ({train_time_stats['after_optimal_pct']:.1f}%)")

print("\n🔬 Dilution Distribution (Training):")
for dil, count in dilution_stats['distribution'].items():
    pct = dilution_stats['percentages'][dil]
    print(f"  {dil}: {count} samples ({pct:.1f}%)")

print("\n" + "=" * 70)

## 10. Recommended Balancing Strategy

In [ ]:
print("=" * 70)
print("RECOMMENDED BALANCING STRATEGY")
print("=" * 70)

print("\n1️⃣ CLASS WEIGHTS (for CrossEntropyLoss):")
print("   Use inverse frequency weights:")
for i in range(5):
    if weights_inverse[i] > 0:
        print(f"      Grade {i+1}: {weights_inverse[i]:.4f}")

print("\n2️⃣ WEIGHTED SAMPLING:")
print("   Use joint weighted sampler with:")
print("      - Grade weight: 0.5 (most important)")
print("      - Dilution weight: 0.3 (ensure both types)")
print("      - Time delta weight: 0.2 (diverse time ranges)")

print("\n3️⃣ AUGMENTATION:")
print("   Apply stronger augmentation to minority classes:")
for grade in sorted(train_stats['distribution'].keys()):
    pct = train_stats['percentages'][grade]
    if pct < 20:
        print(f"      - Grade {grade} ({pct:.1f}%): Strong augmentation")
    elif pct < 30:
        print(f"      - Grade {grade} ({pct:.1f}%): Moderate augmentation")
    else:
        print(f"      - Grade {grade} ({pct:.1f}%): Light augmentation")

print("\n4️⃣ LOSS FUNCTION:")
print("   Multi-task loss:")
print("      - Grade: Weighted CrossEntropyLoss (α = 0.7)")
print("      - Time: Huber Loss with time_delta weighting (β = 0.3)")

print("\n" + "=" * 70)
print("\n✓ Strategy: HYBRID BALANCING")
print("  Combines class weights + joint sampling + targeted augmentation")
print("=" * 70)

## 11. Save Configuration

In [ ]:
# Save analysis results
def convert_to_serializable(obj):
    """Convert numpy/pandas types to native Python types"""
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(item) for item in obj]
    elif hasattr(obj, 'item'):
        return obj.item()
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    else:
        return obj

summary = {
    'dataset_info': {
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'test_samples': len(test_df),
        'total_samples': len(train_df) + len(val_df) + len(test_df)
    },
    'grade_stats': convert_to_serializable(train_stats),
    'time_delta_stats': convert_to_serializable(train_time_stats),
    'dilution_stats': convert_to_serializable(dilution_stats),
    'class_weights': {
        'inverse': weights_inverse.tolist(),
        'effective': weights_effective.tolist()
    },
    'minority_classes': [g for g in train_stats['distribution'].keys() 
                        if train_stats['percentages'][g] < 20],
    'recommended_strategy': {
        'method': 'hybrid',
        'class_weights': 'inverse',
        'sampler': 'joint_weighted',
        'sampler_params': {
            'grade_weight': 0.5,
            'dilution_weight': 0.3,
            'time_delta_weight': 0.2
        },
        'loss_weights': {
            'grade_loss': 0.7,
            'time_loss': 0.3
        }
    },
    'notes': [
        f"Grade {train_stats['max_class']} is dominant ({train_stats['percentages'][train_stats['max_class']]:.1f}% of data)",
        f"Grades {', '.join(map(str, [g for g in train_stats['distribution'].keys() if train_stats['percentages'][g] < 20]))} are minority classes",
        f"Time delta is skewed (mean={train_time_stats['mean']:.2f}, median={train_time_stats['median']:.2f})",
        f"Imbalance ratio: {train_stats['imbalance_ratio']:.2f}x (highly imbalanced)",
        "Recommend using hybrid balancing approach"
    ]
}

output_file = output_dir / 'class_balance_config.json'
with open(output_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"✓ Configuration saved to: {output_file}")
print("\n✓ Analysis complete! Ready to train with balanced approach.")